In [ ]:
!pip install -q ultralytics roboflow timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 125.9 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from ultralytics import YOLO
from ultralytics.models.yolo.detect.train import DetectionTrainer

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
teacher = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True,
    features_only=True
).to(device)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

adapter = nn.Conv2d(768, 256, 1).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [ ]:
class KDTrainer(DetectionTrainer):
    def loss(self, batch, preds=None):
        # Standard YOLO detection loss
        yolo_loss, loss_items = super().loss(batch, preds)

        imgs = batch["img"].to(self.device)

        # ---- Teacher forward ----
        with torch.no_grad():
            t_feat = teacher(imgs)[-1]
            t_feat = adapter(t_feat)

        # ---- Student forward (features only) ----
        _, s_feats = self.model(imgs, return_features=True)

        kd_loss = F.mse_loss(
            s_feats[-1],
            F.interpolate(
                t_feat,
                size=s_feats[-1].shape[-2:],
                mode="bilinear",
                align_corners=False
            )
        )

        total_loss = yolo_loss + 0.3 * kd_loss
        return total_loss, loss_items


In [ ]:
from roboflow import Roboflow

In [ ]:
rf = Roboflow(api_key="zU5zoCd2ANSr1c3NF8P8")      # e.g. "RF0KpN9Wn1QhvwC6Wm4M"
project = rf.workspace("walker-6w6cm").project("weed_detect_yolov11-mu5py")  # workspace + project
version = project.version(2)                         # dataset / model version number
dataset = version.download("yolov8")

print("Dataset path:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to weed_detect_yolov11-2 in yolov8:: 100%|██████████| 2612/2612 [00:00<00:00, 3846.45it/s]

Dataset path: /content/weed_detect_yolov11-2


In [ ]:
yolo = YOLO("/content/best (2).pt")

yolo.train(
    data=f"{dataset.location}/data.yaml",
    epochs=90,
    imgsz=640,
    batch=16,
    trainer=KDTrainer,
    device=device,
    name="yolov11_swin_kd"
)


Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weed_detect_yolov11-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=90, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/best (2).pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11_swin_kd, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abd439cd970>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov_distill_best.pt")
model.val(
    data="/content/weed_detect_yolov11-2/data.yaml"
)



Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2481.0±1176.5 MB/s, size: 122.0 KB)
val: Scanning /content/weed_detect_yolov11-2/valid/labels.cache... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 19.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.7it/s 2.3s
                   all         50        590      0.627      0.596      0.586      0.209
                  crop         39        269        0.6      0.528        0.5      0.148
                  weed         34        321      0.654      0.664      0.672      0.269
Speed: 11.3ms preprocess, 7.5ms inference, 0.0ms loss, 5.4ms postprocess per image
Results saved to /content/runs/detect/val3


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abbd7f0e0f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
baseline_path = "/content/best (2).pt"
distill_path  = "/content/yolov_distill_best.pt"
data_yaml     = "/content/weed_detect_yolov11-2/data.yaml"


In [ ]:
from ultralytics import YOLO

YOLO(baseline_path).val(data=data_yaml)
YOLO(distill_path).val(data=data_yaml)


Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1345.7±368.9 MB/s, size: 111.6 KB)
val: Scanning /content/weed_detect_yolov11-2/valid/labels.cache... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 19.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.8it/s 2.2s
                   all         50        590      0.654      0.578       0.58      0.208
                  crop         39        269      0.553      0.502      0.458      0.125
                  weed         34        321      0.755      0.654      0.702      0.291
Speed: 4.2ms preprocess, 15.6ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to /content/runs/detect/val4
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summar

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abbeabec0b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

tflite comversion

In [ ]:
model = YOLO("yolov_distill_best.pt")

In [ ]:
model.export(format="tflite", imgsz=416, int8=True)


Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'yolov_distill_best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 6, 3549) (6.0 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3,<1.29.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 7.92s
Prepared 11 packages in 7.73s
Installed 11 packages in 252ms
 + ai-edge-litert==2.1.2
 + backports-strenum==1.3.1
 + colorama==0.4.6
 + coloredlogs==15.

Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 1.2s, saved as 'yolov_distill_best.onnx' (11.6 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 43.4files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'yolov_distill_best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 416, 416, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 6, 3549), dtype=tf.float32, name=None)
Captures:
  134949718913680: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  134949718912144: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  134949718912912: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  134949192156368: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  134949718913488: TensorSpec(shape=(3, 3

'yolov_distill_best_saved_model/yolov_distill_best_int8.tflite'